# Phase 2 — Maximum Likelihood Estimation: Exploration Notebook

This notebook explores the MLE fitting pipeline:
- Fit distributions to synthetic salary data
- Examine parameter estimates, standard errors, and confidence intervals
- Verify MLE properties (consistency, asymptotic normality)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.data_gen import generate_salary_data, generate_severance_data
from src.fitting import fit_all, fit_lognormal, fit_normal, fit_pareto

sns.set_theme(style="whitegrid", font_scale=1.1)
SEED = 42

## 1. Fit All Candidates to Salary Data

In [ ]:
salary = generate_salary_data(1000, seed=SEED)
results = fit_all(salary)

rows = []
for r in results:
    rows.append({
        "Distribution": r.distribution,
        "Log-Likelihood": f"{r.loglik:.1f}",
        "AIC": f"{r.aic:.1f}",
        "BIC": f"{r.bic:.1f}",
        "Parameters": str({k: f"{v:.4f}" for k, v in r.params.items()}),
    })

pd.DataFrame(rows)

## 2. LogNormal Fit Details

In [ ]:
result = fit_lognormal(salary)

print("LogNormal MLE Results")
print("=" * 40)
print(f"mu_hat = {result.params['mu']:.4f}")
print(f"sigma_hat = {result.params['sigma']:.4f}")
print(f"Log-likelihood = {result.loglik:.2f}")
print(f"\nStandard Errors:")
print(f"  SE(mu) = {result.se['mu']:.4f}")
print(f"  SE(sigma) = {result.se['sigma']:.4f}")
print(f"\n95% Confidence Intervals:")
print(f"  mu: [{result.ci_lower['mu']:.4f}, {result.ci_upper['mu']:.4f}]")
print(f"  sigma: [{result.ci_lower['sigma']:.4f}, {result.ci_upper['sigma']:.4f}]")
print(f"\nTrue values: mu=9.1, sigma=0.4")
print(f"mu covered: {result.ci_lower['mu'] <= 9.1 <= result.ci_upper['mu']}")
print(f"sigma covered: {result.ci_lower['sigma'] <= 0.4 <= result.ci_upper['sigma']}")

## 3. Visual: Fitted PDF vs Histogram

In [ ]:
from src.distributions import LogNormalDist, NormalDist

best_fit = fit_lognormal(salary)
normal_fit = fit_normal(salary)

lognorm_dist = LogNormalDist(mu=best_fit.params["mu"], sigma=best_fit.params["sigma"])
norm_dist = NormalDist(mu=normal_fit.params["mu"], sigma=normal_fit.params["sigma"])

x = np.linspace(salary.min() * 0.5, np.percentile(salary, 99), 500)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(salary, bins=50, density=True, alpha=0.3, color="gray", label="Data")
ax.plot(x, lognorm_dist.pdf(x), linewidth=2.5, color="#2A9D8F",
        label=f"LogNormal MLE (loglik={best_fit.loglik:.0f})")
ax.plot(x, norm_dist.pdf(x), linewidth=2.5, color="#E63946", linestyle="--",
        label=f"Normal MLE (loglik={normal_fit.loglik:.0f})")
ax.set_xlabel("Salary (R$)")
ax.set_ylabel("Density")
ax.set_title("MLE Fitted Distributions vs Data")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Pareto Fit for Severance Data

In [ ]:
severance = generate_severance_data(200, seed=SEED)
pareto_result = fit_pareto(severance, x_m=10000.0)

print("Pareto MLE for Severance Costs")
print("=" * 40)
print(f"alpha_hat = {pareto_result.params['alpha']:.4f}")
print(f"x_m = {pareto_result.params['x_m']:.0f}")
print(f"SE(alpha) = {pareto_result.se['alpha']:.4f}")
print(f"95% CI: [{pareto_result.ci_lower['alpha']:.4f}, {pareto_result.ci_upper['alpha']:.4f}]")
print(f"\nTrue alpha = 2.5")
print(f"Covered: {pareto_result.ci_lower['alpha'] <= 2.5 <= pareto_result.ci_upper['alpha']}")

## 5. Consistency Check: MLE Converges as n Grows

In [ ]:
sample_sizes = [50, 100, 200, 500, 1000, 5000]
rng = np.random.default_rng(SEED)

fig, ax = plt.subplots(figsize=(10, 6))

for n in sample_sizes:
    mu_hats = []
    for _ in range(200):
        data = rng.lognormal(9.1, 0.4, size=n)
        r = fit_lognormal(data)
        mu_hats.append(r.params["mu"])
    ax.boxplot([mu_hats], positions=[n], widths=n*0.3, showfliers=False)

ax.axhline(9.1, color="red", linestyle="--", label="True mu = 9.1")
ax.set_xscale("log")
ax.set_xlabel("Sample size (n)")
ax.set_ylabel("mu_hat")
ax.set_title("MLE Consistency: mu_hat Converges to True Value")
ax.legend()
plt.tight_layout()
plt.show()